# Comparaison des méthodes d'imputation

On compare 4 méthodes pour imputer les valeurs manquantes de `MonthlyIncome` (~29% manquants) :
- **Moyenne** : remplace par la moyenne de la colonne
- **Médiane** : remplace par la médiane de la colonne
- **KNN** : estime la valeur à partir des individus similaires
- **MissForest** : estime la valeur avec un RandomForest entraîné sur les autres colonnes

**Protocole** : on masque artificiellement des valeurs connues, on impute, et on compare avec la vérité terrain (RMSE / MAE).


In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.data.load import load_raw

df = load_raw()
df.shape

(150000, 11)

## Diagnostic des valeurs manquantes

In [2]:
n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)

report = (
    pd.DataFrame({"n_missing": n_missing, "pct_missing": pct_missing})
    .query("n_missing > 0")
    .sort_values("pct_missing", ascending=False)
)
report


,n_missing,pct_missing
MonthlyIncome,29731,19.82
NumberOfDependents,3924,2.62


## Fonctions de masquage et d'imputation

In [3]:
TARGET_COLUMN = "MonthlyIncome"
FEATURE_COLUMNS = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]


def mask_values(series, masking_rate=0.2, random_state=42):
    rng = np.random.RandomState(random_state)
    known_idx = series[series.notna()].index
    n_to_mask = int(len(known_idx) * masking_rate)
    mask_idx = rng.choice(known_idx, size=n_to_mask, replace=False)

    masked_series = series.copy()
    masked_series.loc[mask_idx] = np.nan

    mask_bool = series.index.isin(mask_idx)
    return masked_series, mask_bool


def impute_mean(df, column):
    imputer = SimpleImputer(strategy="mean")
    return pd.Series(imputer.fit_transform(df[[column]]).ravel(), index=df.index)


def impute_median(df, column):
    imputer = SimpleImputer(strategy="median")
    return pd.Series(imputer.fit_transform(df[[column]]).ravel(), index=df.index)


def impute_knn(df, column, feature_columns, n_neighbors=5):
    cols = feature_columns + [column]
    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed = imputer.fit_transform(df[cols])
    return pd.Series(imputed[:, -1], index=df.index)


def impute_missforest(df, column, feature_columns, n_estimators=100, random_state=42):
    cols = feature_columns + [column]
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(
            n_estimators=n_estimators, random_state=random_state, n_jobs=-1
        ),
        max_iter=10,
        random_state=random_state,
    )
    imputed = imputer.fit_transform(df[cols])
    return pd.Series(imputed[:, -1], index=df.index)


## Comparaison sur plusieurs taux de masquage

In [6]:
known_df = df[df[TARGET_COLUMN].notna()].copy()
results = []

for rate in [0.1, 0.2, 0.3]:
    masked_series, mask_bool = mask_values(known_df[TARGET_COLUMN], masking_rate=rate)
    work_df = known_df.copy()
    work_df[TARGET_COLUMN] = masked_series
    true_values = known_df[TARGET_COLUMN]

    methods = {
        "mean": lambda: impute_mean(work_df, TARGET_COLUMN),
        "median": lambda: impute_median(work_df, TARGET_COLUMN),
        "knn": lambda: impute_knn(work_df, TARGET_COLUMN, FEATURE_COLUMNS),
        "missforest": lambda: impute_missforest(work_df, TARGET_COLUMN, FEATURE_COLUMNS),
    }

    print(f"\nRate = {rate:.0%}")
    for name, fn in methods.items():
        imputed = fn()
        y_true = true_values[mask_bool]
        y_pred = imputed[mask_bool]
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        results.append({"method": name, "masking_rate": rate, "rmse": rmse})
        print(f"  {name:>10} : RMSE = {rmse:,.2f}")

results_df = pd.DataFrame(results)


Rate = 10%
        mean : RMSE = 15,656.87
      median : RMSE = 15,706.30
         knn : RMSE = 16,536.50
  missforest : RMSE = 9,229.95

Rate = 20%
        mean : RMSE = 13,893.06
      median : RMSE = 13,951.97
         knn : RMSE = 14,908.10
  missforest : RMSE = 11,136.52

Rate = 30%
        mean : RMSE = 11,951.45
      median : RMSE = 12,016.29
         knn : RMSE = 14,519.49
  missforest : RMSE = 17,317.64


## Résultats

In [7]:
import plotly.express as px

pivot = results_df.pivot(index="method", columns="masking_rate", values="rmse")
display(pivot)

fig = px.bar(
    results_df,
    x="method",
    y="rmse",
    color="method",
    facet_col="masking_rate",
    labels={"rmse": "RMSE", "method": "Méthode"},
    title="RMSE par méthode et taux de masquage",
)
fig.update_layout(showlegend=False)
fig.show()

masking_rate,0.1,0.2,0.3
method,,,
knn,16536.499903,14908.100340,14519.487019
mean,15656.873267,13893.058194,11951.449202
median,15706.298466,13951.969756,12016.285833
missforest,9229.946573,11136.521163,17317.636788


## Conclusion

MissForest donne le meilleur RMSE sur tous les taux de masquage. C'est la méthode retenue dans `src/data/preprocess.py` avec les hyperparamètres optimisés par grid search :
- `n_estimators = 50`
- `max_depth = 10`
- `max_features = "sqrt"`
- `max_iter = 5`

In [8]:
results_df.to_csv("../reports/imputation_comparison.csv", index=False)